In [ ]:
# ==========================
# 0) LIBRARIES
# ==========================
import os
files = ['train.csv', 'test.csv', 'sample_prediction.csv']
if all(os.path.exists(f) for f in files):
    print('Ready!')
else:
    print('Upload files!')

import sys

!{sys.executable} -m pip -q install xgboost lightgbm

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import lightgbm as lgb
from lightgbm import LGBMRegressor

SEED = 42
np.random.seed(SEED)

In [ ]:
# =========================
# LOAD DATA
# =========================

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
sample = pd.read_csv("sample_prediction.csv")

print(train.shape, test.shape, sample.shape)
train.head()

In [ ]:
# =========================
# CLEAN + NORMALIZE 
# =========================
def normalize_cols(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def drop_index_cols(df):
    df = df.copy()
    for c in ["Unnamed: 0", "index"]:
        if c in df.columns and df[c].nunique(dropna=False) == len(df):
            df = df.drop(columns=[c])
    return df

def normalize_cats(df):
    df = df.copy()
    for c in ["Status", "Sensor Position Side"]:
        if c in df.columns:
            s = df[c].astype("string").str.strip().str.lower()
            s = s.replace({"nan": pd.NA, "none": pd.NA, "": pd.NA})
            df[c] = s
    return df

train = normalize_cols(train)
test  = normalize_cols(test)

train = drop_index_cols(train)
test  = drop_index_cols(test)

train = normalize_cats(train)
test  = normalize_cats(test)

train = train.drop_duplicates()
print("After clean:", train.shape, test.shape)

In [ ]:
# =========================
# FEATURE ENGINEERING 
# =========================
import re

def safe_div(a, b):
    b = np.where(b == 0, np.nan, b)
    return a / b

def sanitize_colnames(cols):
    cols = [str(c) for c in cols]
    cols = [re.sub(r'[^0-9a-zA-Z_]+', '_', c) for c in cols]   # remove special chars
    cols = [re.sub(r'_+', '_', c).strip('_') for c in cols]   # collapse underscores
    return cols

def fe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Strong column sanitization (prevents LightGBM JSON feature-name error)
    df.columns = sanitize_colnames(df.columns)

    def has(cols):
        return all(c in df.columns for c in cols)

    # Tank volume + shape
    if has(["Tank_Width_m", "Tank_Length_m", "Tank_Height_m"]):
        W = df["Tank_Width_m"].astype(float)
        L = df["Tank_Length_m"].astype(float)
        H = df["Tank_Height_m"].astype(float)
        df["tank_vol"] = W * L * H
        df["tank_w_over_l"] = safe_div(W, L)

    # Vapour fraction
    if has(["Vapour_Height_m", "Tank_Height_m"]):
        df["vapour_height_frac"] = safe_div(
            df["Vapour_Height_m"].astype(float),
            df["Tank_Height_m"].astype(float)
        )

    # Obstacle area
    if has(["Obstacle_Width_m", "Obstacle_Height_m"]):
        df["obs_area"] = (
            df["Obstacle_Width_m"].astype(float) *
            df["Obstacle_Height_m"].astype(float)
        )

    # Sensor radius
    if has(["Sensor_Position_x", "Sensor_Position_y", "Sensor_Position_z"]):
        x = df["Sensor_Position_x"].astype(float)
        y = df["Sensor_Position_y"].astype(float)
        z = df["Sensor_Position_z"].astype(float)
        df["sensor_r"] = np.sqrt(x*x + y*y + z*z)

    # Sensor_r / obstacle distance
    if has(["sensor_r", "Obstacle_Distance_to_BLEVE_m"]):
        df["sensor_r_over_obsdist"] = safe_div(
            df["sensor_r"].astype(float),
            df["Obstacle_Distance_to_BLEVE_m"].astype(float)
        )

    # Safety: inf -> nan
    df = df.replace([np.inf, -np.inf], np.nan)

    # Fill numeric vs categorical safely (prevents StringArray fill error)
    num_cols = df.select_dtypes(include=[np.number]).columns
    cat_cols = df.columns.difference(num_cols)

    df[num_cols] = df[num_cols].fillna(0)
    df[cat_cols] = df[cat_cols].fillna("missing")

    return df

train_fe = fe(train)
test_fe  = fe(test)

# Auto-detect target (after sanitization)
possible_targets = ["Target_Pressure_bar", "Target_Pressure_bar_"]
TARGET = None
for t in possible_targets:
    if t in train_fe.columns:
        TARGET = t
        break

if TARGET is None:
    raise KeyError(f"Target column not found. Some cols: {train_fe.columns[:15].tolist()}")

print("Using TARGET:", TARGET)
print("FE shapes:", train_fe.shape, test_fe.shape)

In [ ]:
import numpy as np
from lightgbm import LGBMRegressor
import lightgbm as lgb

# Custom MAPE loss function for LightGBM
def mape_loss(y_true, y_pred):
    """
    Gradient and Hessian for MAPE loss
    MAPE = mean(|y_true - y_pred| / y_true)
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    
    # Avoid division by zero
    epsilon = 1e-6
    y_true_safe = np.where(np.abs(y_true) < epsilon, epsilon, y_true)
    
    # Gradient of MAPE w.r.t. y_pred
    # d/dy_pred [|y_true - y_pred| / y_true] = -sign(y_true - y_pred) / y_true
    diff = y_true - y_pred
    gradient = -np.sign(diff) / y_true_safe
    
    # Hessian (second derivative) - approximate as constant
    hessian = np.ones_like(y_pred) / (np.abs(y_true_safe) + epsilon)
    
    return gradient, hessian

# Better: Use MAE on ORIGINAL scale (not log)
# This naturally minimizes percentage errors
params = {
    "objective": "regression_l1",  # MAE loss
    "metric": "mae",
    "boosting_type": "gbdt",
    "n_estimators": 10000,
    "learning_rate": 0.005,        # SLOWER - more precision
    "num_leaves": 127,              # REDUCED - less overfitting
    "max_depth": 8,                 # NEW - force small trees
    "min_data_in_leaf": 30,         # INCREASED - smoother predictions
    "min_sum_hessian_in_leaf": 0.01,
    "feature_fraction": 0.75,       # REDUCED
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "reg_alpha": 0.5,               # L1 regularization
    "reg_lambda": 3.0,              # L2 regularization (INCREASED)
    "random_state": 42,
    "bagging_seed": 42,
    "feature_fraction_seed": 42,
    "data_random_seed": 42,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True,
    "deterministic": True,
}

# KEY CHANGE: Don't use log1p transformation!
# Train directly on original scale
y_train_original = train_fe[TARGET].values  # NOT log1p
X_train_original = train_fe.drop(columns=[TARGET])

# Train model
model = LGBMRegressor(**params)
model.fit(
    X_train_original, y_train_original,
    eval_set=[(X_val_original, y_val_original)],
    eval_metric="mae",
    callbacks=[
        lgb.early_stopping(300),
        lgb.log_evaluation(0)
    ]
)

# Predict directly (no expm1 needed)
test_pred = model.predict(X_test)
test_pred = np.clip(test_pred, 1e-6, None)  # Ensure positive

In [ ]:
# Calculate pressure quantiles
p5 = y_train_original.quantile(0.05)
p25 = y_train_original.quantile(0.25)
p75 = y_train_original.quantile(0.75)

print(f"Pressure segments:")
print(f"  Near-zero (0-5%): 0 to {p5:.4f}")
print(f"  Low (5-25%): {p5:.4f} to {p25:.4f}")
print(f"  Medium (25-75%): {p25:.4f} to {p75:.4f}")
print(f"  High (75-100%): {p75:.4f}+")

# Create segment masks
seg_near_zero = y_train_original <= p5
seg_low = (y_train_original > p5) & (y_train_original <= p25)
seg_medium = (y_train_original > p25) & (y_train_original <= p75)
seg_high = y_train_original > p75

segments = {
    'near_zero': (seg_near_zero, p5, p25, p75),
    'low': (seg_low, p5, p25, p75),
    'medium': (seg_medium, p5, p25, p75),
    'high': (seg_high, p5, p25, p75)
}

# Train separate models with DIFFERENT hyperparameters
segment_models = {}
segment_scores = {}

for seg_name, (seg_mask, _, _, _) in segments.items():
    print(f"\n=== Training {seg_name} segment ===")
    print(f"Samples: {seg_mask.sum()}")
    
    if seg_mask.sum() < 10:
        print(f"Skipping {seg_name} - too few samples")
        continue
    
    X_seg = X_train_original[seg_mask]
    y_seg = y_train_original[seg_mask]
    
    # Adapt hyperparameters per segment
    if seg_name == 'near_zero':
        # Near-zero needs HIGH regularization (conservative predictions)
        seg_params = params.copy()
        seg_params.update({
            "num_leaves": 63,
            "min_data_in_leaf": 50,
            "reg_lambda": 5.0,
            "learning_rate": 0.003,
        })
    elif seg_name == 'high':
        # High-pressure needs flexibility (captures variance)
        seg_params = params.copy()
        seg_params.update({
            "num_leaves": 200,
            "min_data_in_leaf": 10,
            "reg_lambda": 1.0,
            "learning_rate": 0.01,
        })
    else:
        # Medium segments: balanced
        seg_params = params.copy()
    
    model_seg = LGBMRegressor(**seg_params)
    model_seg.fit(
        X_seg, y_seg,
        eval_set=[(X_val_original[seg_mask], y_val_original[seg_mask])] 
                if seg_mask.sum() > 0 else None,
        eval_metric="mae",
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(0)
        ]
    )
    
    segment_models[seg_name] = model_seg
    
    # Evaluate on validation
    if seg_mask.sum() > 0:
        y_val_seg = y_val_original[seg_mask]
        pred_val_seg = model_seg.predict(X_val_original[seg_mask])
        mape_seg = np.mean(np.abs((y_val_seg - pred_val_seg) / (y_val_seg + 1e-6)))
        segment_scores[seg_name] = mape_seg
        print(f"Validation MAPE: {mape_seg:.5f}")

print("\n" + "="*50)
print("Segment Validation MAPE Scores:")
for seg_name, score in segment_scores.items():
    print(f"  {seg_name}: {score:.5f}")
print("="*50)

In [ ]:
def build_risk_weighted_ensemble(fold_predictions, fold_y_vals, segment_thresholds):
    """
    Weight each fold's predictions by accuracy on critical low-pressure zone
    """
    p5, p25, p75 = segment_thresholds
    
    fold_weights = []
    fold_mape_by_segment = {}
    
    for fold_idx, (fold_pred, fold_y) in enumerate(zip(fold_predictions, fold_y_vals)):
        # Weight breakdown by segment
        masks = {
            'near_zero': fold_y <= p5,
            'low': (fold_y > p5) & (fold_y <= p25),
            'medium': (fold_y > p25) & (fold_y <= p75),
            'high': fold_y > p75
        }
        
        mapes = {}
        total_weighted_mape = 0
        
        for seg_name, mask in masks.items():
            if mask.sum() > 0:
                seg_mape = np.mean(np.abs(
                    (fold_y[mask] - fold_pred[mask]) / (fold_y[mask] + 1e-6)
                ))
                mapes[seg_name] = seg_mape
                
                # Weight near-zero and low segments more heavily (2x and 1.5x)
                if seg_name == 'near_zero':
                    total_weighted_mape += seg_mape * 2.0
                elif seg_name == 'low':
                    total_weighted_mape += seg_mape * 1.5
                else:
                    total_weighted_mape += seg_mape * 1.0
            else:
                mapes[seg_name] = 0
        
        # Inverse MAPE as weight (better fold = higher weight)
        fold_weight = 1.0 / (1.0 + total_weighted_mape)
        fold_weights.append(fold_weight)
        fold_mape_by_segment[fold_idx] = mapes
        
        print(f"\nFold {fold_idx}:")
        for seg_name, mape in mapes.items():
            print(f"  {seg_name}: {mape:.5f}")
        print(f"  Weight: {fold_weight:.4f}")
    
    # Normalize weights to sum to 1
    fold_weights = np.array(fold_weights)
    fold_weights /= fold_weights.sum()
    
    print(f"\nFold weights (normalized): {fold_weights}")
    
    # Weighted ensemble
    ensemble_pred = np.average(
        np.stack(fold_predictions, axis=0),
        axis=0,
        weights=fold_weights
    )
    
    return ensemble_pred, fold_weights

# Use in CV loop:
test_preds_folds = []
fold_y_vals = []

for fold, (tr, va) in enumerate(gkf.split(X_train_original, y_train_original, groups), 1):
    print(f"--- Fold {fold} ---")
    
    Xtr, Xva = X_train_original.iloc[tr], X_train_original.iloc[va]
    ytr, yva = y_train_original.iloc[tr], y_train_original.iloc[va]
    
    model = LGBMRegressor(**params)
    model.fit(
        Xtr, ytr,
        eval_set=[(Xva, yva)],
        eval_metric="mae",
        callbacks=[lgb.early_stopping(300), lgb.log_evaluation(0)]
    )
    
    test_pred_fold = model.predict(X_test)
    test_pred_fold = np.clip(test_pred_fold, 1e-6, None)
    
    test_preds_folds.append(test_pred_fold)
    fold_y_vals.append(yva.values)

# Apply risk-aware weighting
p5 = y_train_original.quantile(0.05)
p25 = y_train_original.quantile(0.25)
p75 = y_train_original.quantile(0.75)

test_pred_final, fold_weights = build_risk_weighted_ensemble(
    test_preds_folds,
    fold_y_vals,
    (p5, p25, p75)
)

test_pred_final = np.clip(test_pred_final, 1e-6, None)

In [ ]:
# =========================
# FINAL SUMMARY (UPDATED FOR MULTI-SEED)
# =========================
# Create submission
sub = sample.copy()
target_col = sample.columns[1]
sub[target_col] = test_pred_final.astype(float)

out_path = "prediction.csv"
sub.to_csv(out_path, index=False)

print(f"\n{'='*60}")
print(f"SUBMISSION SAVED TO: {out_path}")
print(f"{'='*60}")
print(f"Test predictions stats:")
print(f"  Min:    {test_pred_final.min():.6f}")
print(f"  Max:    {test_pred_final.max():.6f}")
print(f"  Mean:   {test_pred_final.mean():.6f}")
print(f"  Median: {np.median(test_pred_final):.6f}")
print(f"\nTrain target stats:")
print(f"  Min:    {y_train_original.min():.6f}")
print(f"  Max:    {y_train_original.max():.6f}")
print(f"  Mean:   {y_train_original.mean():.6f}")
print(f"  Median: {y_train_original.median():.6f}")